# 04 — GridSearchCV de los tres candidatos
Comparamos **30 configuraciones por familia**, 90 en total y 450 ajustes de CV. Cada mejor configuración se reajusta con todo entrenamiento (tres ajustes adicionales). Test no se evalúa.

Las cuadrículas fueron acordadas antes del tuning. Conservamos 173 predictores, imputación mediana y escalado solo en SVR. No se añaden transformaciones ni selección de variables durante esta búsqueda.


In [1]:
from pathlib import Path
import sys, json, hashlib
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.pipeline import construir_pipeline
from src.splitting import folds_por_grupos
config = json.loads((ROOT / "configs/pipelines.json").read_text())
snapshot = Path(json.loads((ROOT / "configs/dataset.local.json").read_text())["snapshot"]).name
carpeta = ROOT / "data/modelado" / snapshot
manifest = json.loads((carpeta / "manifest.json").read_text())
split_manifest = json.loads((carpeta / "split/manifest.json").read_text())
for archivo, esperado in [(carpeta / "datos_modelado.parquet", split_manifest["source_sha256"]),
                          (carpeta / "split/asignacion.parquet", split_manifest["assignment_sha256"])]:
    assert hashlib.sha256(archivo.read_bytes()).hexdigest() == esperado, f"Archivo modificado: {archivo.name}"
datos = pd.read_parquet(carpeta / "datos_modelado.parquet")
asignacion = pd.read_parquet(carpeta / "split/asignacion.parquet")
assert datos.index.equals(asignacion.index)
features = manifest["predictors"]
train = datos.loc[asignacion["PARTICION"].eq("train")].copy()
X_train = train[features]
y_train = train[manifest["target"]]
grupos_train = train[manifest["group"]]
solo_train = asignacion.loc[train.index, "ENTRENAMIENTO_OBLIGATORIO"]
print("Entrenamiento:", X_train.shape, "| Test reservado, sin analizar")


Entrenamiento: (3670, 173) | Test reservado, sin analizar


## 1. Cuadrículas y justificación
| Familia | Parámetros | Motivo |
|---|---|---|
| Random Forest | profundidad 5, 10 o ilimitada; hoja mínima 1, 3, 5, 10 o 20; fracción de features 0.5 o 1.0 | Reducir complejidad y correlación entre árboles para controlar la brecha train/CV. Se mantienen 100 árboles por defecto. |
| XGBoost | profundidad 2, 4 o 6; tasa 0.01, 0.03, 0.1, 0.2 o 0.3; 100 o 300 árboles | Explorar capacidad y combinar tasas pequeñas con más árboles. Tasas de aproximadamente distintos órdenes de magnitud y pasos adicionales en la zona alta. |
| SVR | C de 0.1 a 1000 por potencias de 10; gamma scale, 0.001 o 0.01; epsilon 0.1 o 0.3 | Explorar regularización en escala logarítmica, alcance del kernel y tolerancia de error en pKd. |

Elegimos GridSearchCV porque el espacio es pequeño y explícito, y permite evaluar exhaustivamente las 30 combinaciones de cada familia. Los parámetros restantes permanecen por defecto, salvo semilla y paralelismo. La grilla no garantiza incluir el óptimo global.


In [2]:
from sklearn.model_selection import ParameterGrid
from joblib import cpu_count
from src.experimentos import ajustar_candidatos

grids = {
    "RandomForestRegressor": {"modelo__max_depth":[5,10,None], "modelo__min_samples_leaf":[1,3,5,10,20], "modelo__max_features":[0.5,1.0]},
    "XGBRegressor": {"modelo__max_depth":[2,4,6], "modelo__learning_rate":[0.01,0.03,0.1,0.2,0.3], "modelo__n_estimators":[100,300]},
    "SVR": {"modelo__C":[0.1,1,10,100,1000], "modelo__gamma":["scale",0.001,0.01], "modelo__epsilon":[0.1,0.3]},
}
assert all(len(ParameterGrid(g)) == 30 for g in grids.values())
N_JOBS = min(8, cpu_count())
print("Combinaciones:", {m:len(ParameterGrid(g)) for m,g in grids.items()})
print("Trabajos paralelos:", N_JOBS, "| Cada estimador usa un solo core para evitar paralelismo anidado")


Combinaciones: {'RandomForestRegressor': 30, 'XGBRegressor': 30, 'SVR': 30}
Trabajos paralelos: 8 | Cada estimador usa un solo core para evitar paralelismo anidado


## 2. Reutilizar y verificar exactamente los folds
Solo se regeneran los mismos índices y se contrastan con el archivo guardado en 03. Los 293 casos obligatorios no entran a validación. Los scores de búsqueda se llaman `test` en `cv_results_` por convención de sklearn, pero aquí significan **validación de cada fold**, nunca el holdout reservado.


In [3]:
folds = folds_por_grupos(grupos_train, solo_train, n_splits=config["cv_folds"])
base = ROOT / "reports/baseline" / snapshot
contexto_baseline = json.loads((base / "contexto.json").read_text())
assert contexto_baseline["protocol"] == "defaults_v2"
assert contexto_baseline["source_sha256"] == split_manifest["source_sha256"]
assert contexto_baseline["split_assignment_sha256"] == split_manifest["assignment_sha256"]
assert hashlib.sha256((base / "folds.csv").read_bytes()).hexdigest() == contexto_baseline["folds_sha256"]
regenerados = pd.concat([pd.DataFrame({"fila_modelo":X_train.index[va],"fold_validacion":i+1}) for i,(_,va) in enumerate(folds)],ignore_index=True)
pd.testing.assert_frame_equal(regenerados,pd.read_csv(base / "folds.csv"))
for tr, va in folds:
    assert not set(grupos_train.iloc[tr]) & set(grupos_train.iloc[va])
    assert not solo_train.iloc[va].any()
baseline = pd.read_csv(base / "comparacion.csv", index_col="modelo")
assert set(baseline.drop(index="DummyMean").sort_values("rmse_mean").head(3).index) == set(grids)
contexto = {**contexto_baseline, "protocol":"gridsearch_v1", "baseline_file_sha256":hashlib.sha256((base / "comparacion.csv").read_bytes()).hexdigest(),
            "experiments_code_sha256":hashlib.sha256((ROOT / "src/experimentos.py").read_bytes()).hexdigest(),
            "grid":grids, "n_jobs":N_JOBS}
CARPETA_TUNING = ROOT / "reports/tuning" / snapshot
print("Verificados cinco folds y los tres candidatos; test no utilizado.")


Verificados cinco folds y los tres candidatos; test no utilizado.


## 3. Ejecutar y registrar
Esta celda puede tardar varios minutos. Crea tres corridas padre y 90 hijas en MLflow. Se registran resultados train/CV, brechas, tiempos, parámetros y trazabilidad; el mejor pipeline de cada familia se guarda con firma y ejemplo de entrada. Las corridas hijas se registran cuando termina la búsqueda de su familia. Reejecutar crea nuevas corridas y actualiza los reportes locales.


In [4]:
resumen_tuning = ajustar_candidatos(X_train, y_train, folds, grids, CARPETA_TUNING,
                                      contexto, baseline, seed=config["seed"], n_jobs=N_JOBS)
display(resumen_tuning[["modelo","rmse_baseline","rmse_cv","rmse_cv_std","rmse_train","gap_rmse","mejora_rmse","supera_variacion_folds","search_seconds"]].round(4))


RandomForestRegressor: 30 combinaciones × 5 folds


Fitting 5 folds for each of 30 candidates, totalling 150 fits


2026/11/19 02:54:23 INFO mlflow.tracking._tracking_service.client: 🏃 View run config_01 at: http://mlflow:5000/#/experiments/1/runs/3e24f72161664d1d828ccfd23668e6b7.


2026/11/19 02:54:23 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


2026/11/19 02:54:23 INFO mlflow.tracking._tracking_service.client: 🏃 View run config_02 at: http://mlflow:5000/#/experiments/1/runs/ca948cbc024d4e989db9303f474a4ca9.


2026/11/19 02:54:23 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


... (186 outputs left out)

SVR terminado: RMSE CV=1.6249


2026/11/19 02:59:30 INFO mlflow.tracking._tracking_service.client: 🏃 View run tuning_SVR at: http://mlflow:5000/#/experiments/1/runs/0e758da707144452ae97fd5c2c5fa8cc.


2026/11/19 02:59:30 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


,modelo,rmse_baseline,rmse_cv,rmse_cv_std,rmse_train,gap_rmse,mejora_rmse,supera_variacion_folds,search_seconds
0,SVR,1.6503,1.6249,0.0880,0.3709,1.2540,0.0254,False,220.6001
1,XGBRegressor,1.7067,1.6270,0.0954,0.5394,1.0875,0.0798,False,59.7139
2,RandomForestRegressor,1.6325,1.6325,0.1136,0.4862,1.1463,0.0000,False,296.3383


## 4. Interpretación antes de selección final
`mejora_rmse = RMSE_baseline − RMSE_tuning`: positivo significa reducción de error. Siguiendo el PSet, solo marcamos mejora por encima de la variación cuando supera la mayor desviación entre folds de ambas versiones (criterio conservador; no es un test de significancia). Usamos desviación muestral `ddof=1`, igual que el baseline.

El mejor score de CV está optimizado sobre estos mismos folds; no es una estimación final independiente. Los modelos aún no son champion/challenger. En el cuaderno 05 se decidirán los dos finalistas con evidencia y se evaluarán una sola vez sobre test. La similitud desconocida de los 293 casos exclusivos de entrenamiento sigue siendo una limitación.


In [5]:
display(resumen_tuning[["modelo", "best_params", "pipeline_uri"]])


,modelo,best_params,pipeline_uri
0,SVR,"{""modelo__C"": 10, ""modelo__epsilon"": 0.3, ""mod...",runs:/72fe83523ad74823a97ba08087442c50/pipeline
1,XGBRegressor,"{""modelo__learning_rate"": 0.03, ""modelo__max_d...",runs:/ad23c738cbcb43a488fb85ed7db2dc4b/pipeline
2,RandomForestRegressor,"{""modelo__max_depth"": null, ""modelo__max_featu...",runs:/c13cf19ae3264d7db6d1b9688ee05bc4/pipeline
